In [23]:
!apt-get update
!apt-get install -y yosys iverilog
!yosys --version
!iverilog -V

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
yosys is already the newest version (0.9-2)

In [24]:
!wget -O nangate45.lib https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/lib/NangateOpenCellLibrary_typical.lib
!ls -lh nangate45.lib

--2026-04-18 12:07:15--  https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/lib/NangateOpenCellLibrary_typical.lib
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6692032 (6.4M) [text/plain]
Saving to: ‘nangate45.lib’

nangate45.lib       100%[===================>]   6.38M  31.2MB/s    in 0.2s    

2026-04-18 12:07:15 (31.2 MB/s) - ‘nangate45.lib’ saved [6692032/6692032]

-rw-r--r-- 1 root root 6.4M Apr 18 12:07 nangate45.lib


In [25]:
!git clone https://github.com/FCHXWH823/Verilog-Adders.git
!find Verilog-Adders -iname "RCA8.v" -o -iname "KSA8.v"

fatal: destination path 'Verilog-Adders' already exists and is not an empty directory.
Verilog-Adders/Carry Ripple Adder/RCA8.v
Verilog-Adders/Kogge-Stone Adder/KSA8.v


In [26]:
!cp Verilog-Adders/*/RCA8.v .
!cp Verilog-Adders/*/KSA8.v .
!ls

area_candidate_10.v	best_adder_delay.v    nangate45.lib
area_candidate_1.v	constraints.sdc       optimization_log_area.json
area_candidate_2.v	delay_candidate_10.v  optimization_log_delay.json
area_candidate_3.v	delay_candidate_1.v   optimize_adder.py
area_candidate_4.v	delay_candidate_2.v   plot_ppa.py
area_candidate_5.v	delay_candidate_3.v   __pycache__
area_candidate_6.v	delay_candidate_4.v   rca8_generated.v
area_candidate_7.v	delay_candidate_5.v   RCA8.v
area_candidate_8.v	delay_candidate_6.v   run_yosys.py
area_candidate_9.v	delay_candidate_7.v   sample_data
balanced_candidate_1.v	delay_candidate_8.v   synth_adder.ys
balanced_candidate_2.v	delay_candidate_9.v   temp_synth.ys
balanced_candidate_3.v	equiv_check.ys	      testbench_final.v
best_adder_area.v	KSA8.v		      Verilog-Adders


In [27]:
%%writefile constraints.sdc
create_clock -name clk -period 2.0
set_input_delay 0.2 -clock clk [all_inputs]
set_output_delay 0.2 -clock clk [all_outputs]

Overwriting constraints.sdc


In [28]:
%%writefile synth_adder.ys
read_verilog $env(ADDER_FILE)
hierarchy -check -top $env(TOP_MODULE)

flatten

proc; opt; fsm; opt; memory; opt
techmap; opt
dfflibmap -liberty nangate45.lib
abc -liberty nangate45.lib -constr constraints.sdc
clean
stat -liberty nangate45.lib

Overwriting synth_adder.ys


In [29]:
%%writefile run_yosys.py
import subprocess, re, json, sys

def synthesize(verilog_file, top_module, lib_file='nangate45.lib'):
    script = f"""
read_verilog {verilog_file}
hierarchy -check -top {top_module}

flatten

proc; opt; fsm; opt; memory; opt
techmap; opt
dfflibmap -liberty {lib_file}
abc -liberty {lib_file} -constr constraints.sdc
clean
stat -liberty {lib_file}
"""
    with open('temp_synth.ys', 'w') as f:
        f.write(script)

    result = subprocess.run(
        ['yosys', '-s', 'temp_synth.ys'],
        capture_output=True,
        text=True
    )

    log = result.stdout + result.stderr

    if result.returncode != 0:
        raise RuntimeError(log)

    return parse_stats(log)

def parse_stats(log):
    ppa = {}

    m = re.search(r'Chip area for.*?:\s+([\d.]+)', log)
    ppa['area_um2'] = float(m.group(1)) if m else None

    m = re.search(r'Number of cells:\s+(\d+)', log)
    ppa['cell_count'] = int(m.group(1)) if m else None

    m = re.search(r'Longest topological path.*?\((\d+) levels?\)', log, re.S)
    ppa['logic_levels'] = int(m.group(1)) if m else None

    return ppa

if __name__ == '__main__':
    ppa = synthesize(sys.argv[1], sys.argv[2])
    print(json.dumps(ppa, indent=2))

Overwriting run_yosys.py


In [30]:
!python run_yosys.py RCA8.v RCA8

{
  "area_um2": 69.426,
  "cell_count": 37,
  "logic_levels": null
}


In [31]:
!python run_yosys.py KSA8.v KSA8

{
  "area_um2": 70.224,
  "cell_count": 41,
  "logic_levels": null
}


In [32]:
%%writefile optimize_adder.py
import json, os, sys
from openai import OpenAI
from run_yosys import synthesize

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def get_mode_prompt(mode):
    common = """You are an expert digital circuit designer.

Generate ONLY valid synthesizable Verilog for an 8-bit ripple-carry adder.

Strict requirements:
1. Keep the top-level module name exactly RCA8.
2. Include the full FA submodule definition in the SAME file.
3. The file must contain BOTH:
   - module FA ... endmodule
   - module RCA8 ... endmodule
4. RCA8 must instantiate FA submodules structurally.
5. Use wire [7:1] c for the carry chain.
6. fa0 handles bit 0 with cin = 1'b0 and cout = c[1].
7. The generate loop only handles bits 1 through 6.
8. fa7 handles bit 7 and drives cout.
9. Never use c[0] or c[8].
10. Return ONLY valid Verilog code.
11. No markdown fences, no explanation.
"""

    if mode == "area":
        return common + """
Optimization goal:
- Minimize cell count and area.
- Delay target is relaxed (about 14 logic levels).
"""
    elif mode == "delay":
        return common + """
Optimization goal:
- Prefer lower delay if possible.
- Target about 6 logic levels, even if area increases somewhat.
"""
    elif mode == "balanced":
        return common + """
Optimization goal:
- Balanced PPA.
- Target about 10 logic levels if feasible while also minimizing cell count.
"""

def clean_verilog(verilog_text):
    verilog_text = verilog_text.replace("```verilog", "").replace("```", "").strip()
    start = verilog_text.find("module")
    end = verilog_text.rfind("endmodule")
    if start != -1 and end != -1:
        verilog_text = verilog_text[start:end + len("endmodule")]
    return verilog_text.strip()

def llm_generate(system_prompt, history):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}] + history
    )
    return clean_verilog(response.choices[0].message.content)

def build_feedback_prompt(mode, iteration, ppa, best_ppa):
    if mode == "area":
        goal = "Reduce cell count and area. Delay can be relaxed up to about 14 logic levels."
    elif mode == "delay":
        goal = "Reduce delay aggressively toward about 6 logic levels, even if area increases somewhat."
    else:
        goal = "Find a balanced design: about 10 logic levels if possible, while also reducing cell count."

    return f"""Iteration {iteration} synthesis results:
- Cell count: {ppa['cell_count']}
- Area (um^2): {ppa['area_um2']}
- Logic levels: {ppa['logic_levels']}

Best so far:
- Cell count: {best_ppa['cell_count']}
- Area (um^2): {best_ppa['area_um2']}
- Logic levels: {best_ppa['logic_levels']}

Goal for next iteration:
{goal}

Preserve functional correctness and return ONLY valid Verilog code.
"""

def is_better(mode, ppa, best_ppa):
    if ppa["cell_count"] is None or ppa["area_um2"] is None:
        return False

    if mode == "area":
        return (
            ppa["cell_count"] < best_ppa["cell_count"] or
            (ppa["cell_count"] == best_ppa["cell_count"] and ppa["area_um2"] < best_ppa["area_um2"])
        )

    if mode == "delay":
        # logic_levels may be None in this environment, so fall back to area as secondary proxy
        if ppa["logic_levels"] is not None and best_ppa["logic_levels"] is not None:
            return (
                ppa["logic_levels"] < best_ppa["logic_levels"] or
                (ppa["logic_levels"] == best_ppa["logic_levels"] and ppa["area_um2"] < best_ppa["area_um2"])
            )
        return ppa["area_um2"] < best_ppa["area_um2"]

    if mode == "balanced":
        # prefer lower logic_levels when available, otherwise use cells then area
        if ppa["logic_levels"] is not None and best_ppa["logic_levels"] is not None:
            return (
                ppa["logic_levels"] < best_ppa["logic_levels"] or
                (ppa["logic_levels"] == best_ppa["logic_levels"] and ppa["cell_count"] < best_ppa["cell_count"])
            )
        return (
            ppa["cell_count"] < best_ppa["cell_count"] or
            (ppa["cell_count"] == best_ppa["cell_count"] and ppa["area_um2"] < best_ppa["area_um2"])
        )

    return False

def run_loop(baseline_file, top_module, mode, max_iter=10):
    system_prompt = get_mode_prompt(mode)

    with open(baseline_file, "r") as f:
        baseline_verilog = f.read()

    history = [{
        "role": "user",
        "content": f"""Here is my current 8-bit adder design:

{baseline_verilog}

Optimize this design in {mode} mode.
Preserve functional correctness and keep the top module name {top_module}.
Your first proposal may be identical."""
    }]

    best_ppa = {"cell_count": 10**9, "area_um2": 10**9, "logic_levels": 10**9}
    best_code = baseline_verilog
    results = []

    for i in range(1, max_iter + 1):
        print(f"\n=== Iteration {i} ===")
        verilog = llm_generate(system_prompt, history)

        fname = f"{mode}_candidate_{i}.v"
        with open(fname, "w") as f:
            f.write(verilog)

        try:
            ppa = synthesize(fname, top_module)
        except Exception as e:
            print("Synthesis failed:", e)
            history.append({"role": "assistant", "content": verilog})
            history.append({"role": "user", "content": "Synthesis failed. Fix syntax, module interface, and structural issues. Return valid Verilog only."})
            continue

        print(f"Cells: {ppa['cell_count']}, Area: {ppa['area_um2']}, Levels: {ppa['logic_levels']}")
        results.append({"iteration": i, "ppa": ppa, "file": fname})

        if is_better(mode, ppa, best_ppa):
            best_ppa = ppa
            best_code = verilog
            print("*** New best! ***")

        history.append({
    "role": "user",
    "content": """Synthesis failed because your RCA8 carry chain is out of bounds.
You referenced an invalid carry index such as c[8].

Fix it with this exact structure:
- wire [7:1] c;
- fa0 handles bit 0 and drives c[1]
- generate loop only for bits 1 through 6
- fa7 handles bit 7 and drives cout
- never use c[8]
Return only valid Verilog."""
})

    best_name = f"best_adder_{mode}.v"
    log_name = f"optimization_log_{mode}.json"

    with open(best_name, "w") as f:
        f.write(best_code)

    with open(log_name, "w") as f:
        json.dump({"mode": mode, "best_ppa": best_ppa, "iterations": results}, f, indent=2)

    print(f"\nBest ({mode}):", best_ppa)

if __name__ == "__main__":
    baseline_file = sys.argv[1]
    top_module = sys.argv[2]
    mode = sys.argv[3]
    run_loop(baseline_file, top_module, mode, max_iter=10)

Overwriting optimize_adder.py


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [34]:
!python optimize_adder.py rca8_generated.v RCA8 area


=== Iteration 1 ===
Cells: 37, Area: 69.426, Levels: None
*** New best! ***

=== Iteration 2 ===
Cells: 37, Area: 53.466, Levels: None
*** New best! ***

=== Iteration 3 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 4 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 5 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 6 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 7 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 8 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 9 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 10 ===
Cells: 37, Area: 53.466, Levels: None

Best (area): {'area_um2': 53.466, 'cell_count': 37, 'logic_levels': None}


In [35]:
!python optimize_adder.py rca8_generated.v RCA8 delay
!python optimize_adder.py rca8_generated.v RCA8 balanced


=== Iteration 1 ===
Cells: 37, Area: 69.426, Levels: None
*** New best! ***

=== Iteration 2 ===
Cells: 37, Area: 53.466, Levels: None
*** New best! ***

=== Iteration 3 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 4 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 5 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 6 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 7 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 8 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 9 ===
Cells: 37, Area: 53.466, Levels: None

=== Iteration 10 ===
Cells: 37, Area: 53.466, Levels: None

Best (delay): {'area_um2': 53.466, 'cell_count': 37, 'logic_levels': None}

=== Iteration 1 ===
Cells: 37, Area: 69.426, Levels: None
*** New best! ***

=== Iteration 2 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 3 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 4 ===
Cells: 37, Area: 69.426, Levels: None

=== Iteration 5 ===
Cells: 37, Area: 69.42

In [45]:
%%writefile plot_ppa.py
import json
import matplotlib.pyplot as plt

with open('optimization_log_delay.json') as f:
    log = json.load(f)

iters = [r['iteration'] for r in log['iterations']]
cells = [r['ppa']['cell_count'] for r in log['iterations']]
areas = [r['ppa']['area_um2'] for r in log['iterations']]

plt.figure(figsize=(8,4))
plt.plot(iters, cells, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Cell count')
plt.title('Cell Count Trajectory')
plt.grid(True)
plt.savefig('cell_trajectory.pdf')
plt.show()

plt.figure(figsize=(8,4))
plt.plot(iters, areas, marker='s')
plt.xlabel('Iteration')
plt.ylabel('Area (um^2)')
plt.title('Area Trajectory')
plt.grid(True)
plt.savefig('area_trajectory.pdf')
plt.show()

Overwriting plot_ppa.py


In [46]:
!python plot_ppa.py

Figure(800x400)
Figure(800x400)


In [38]:
!iverilog -o opt_sim best_adder_balanced.v testbench_final.v
!vvp opt_sim

Test 1: a = 00000000, b = 00000000, sum = 00000000, cout = 0
✓ Sum: 00000000 (expected: 00000000), Cout: 0 (expected: 0)
Test 2: a = 11111111, b = 11111111, sum = 11111110, cout = 1
✓ Sum: 11111110 (expected: 11111110), Cout: 1 (expected: 1)
Test 3: a = 01010101, b = 10101010, sum = 11111111, cout = 0
✓ Sum: 11111111 (expected: 11111111), Cout: 0 (expected: 0)
Test 4: a = 11111111, b = 00000000, sum = 11111111, cout = 0
✓ Sum: 11111111 (expected: 11111111), Cout: 0 (expected: 0)
Test 5: a = 00000001, b = 00000001, sum = 00000010, cout = 0
✓ Sum: 00000010 (expected: 00000010), Cout: 0 (expected: 0)
Test 6: a = 00000000, b = 00000001, sum = 00000001, cout = 0
✓ Sum: 00000001 (expected: 00000001), Cout: 0 (expected: 0)
Test 7: a = 11001100, b = 00110011, sum = 11111111, cout = 0
✓ Sum: 11111111 (expected: 11111111), Cout: 0 (expected: 0)
Test 8: a = 11111110, b = 00000001, sum = 11111111, cout = 0
✓ Sum: 11111111 (expected: 11111111), Cout: 0 (expected: 0)
Test 9: a = 10000000, b = 100000

In [39]:
%%writefile equiv_check.ys
read_verilog -sv rca8_generated.v
rename RCA8 gold

read_verilog -sv -ignore_redef best_adder_balanced.v
rename RCA8 gate

equiv_make gold gate equiv
prep -top equiv
equiv_simple
equiv_status

Overwriting equiv_check.ys


In [40]:
!yosys -s equiv_check.ys


 /----------------------------------------------------------------------------\
 |                                                                            |
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |                                                                            |
 |  Copyright (C) 2012 - 2019  Clifford Wolf <clifford@clifford.at>           |
 |                                                                            |
 |  Permission to use, copy, modify, and/or distribute this software for any  |
 |  purpose with or without fee is hereby granted, provided that the above    |
 |  copyright notice and this permission notice appear in all copies.         |
 |                                                                            |
 |  THE SOFTWARE IS PROVIDED "AS IS" AND THE AUTHOR DISCLAIMS ALL WARRANTIES  |
 |  WITH REGARD TO THIS SOFTWARE INCLUDING ALL IMPLIED WARRANTIES OF          |
 |  MERCHANTABILITY AND FITNESS. IN NO 

In [47]:
%%writefile plot_ppa_all.py
import json
import os
import matplotlib.pyplot as plt

def plot_one(log_file, mode_name):
    with open(log_file, "r") as f:
        log = json.load(f)

    iters = [r["iteration"] for r in log["iterations"] if "ppa" in r]
    cells = [r["ppa"]["cell_count"] for r in log["iterations"] if "ppa" in r]
    areas = [r["ppa"]["area_um2"] for r in log["iterations"] if "ppa" in r]
    levels = [r["ppa"].get("logic_levels") for r in log["iterations"] if "ppa" in r]

    has_levels = any(v is not None for v in levels)

    if has_levels:
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(8, 7))

        ax1.plot(iters, cells, marker='o')
        ax1.set_ylabel("Cell count")
        ax1.grid(True)

        ax2.plot(iters, areas, marker='s')
        ax2.set_ylabel("Area (um^2)")
        ax2.grid(True)

        ax3.plot(iters, levels, marker='^')
        ax3.set_ylabel("Logic levels")
        ax3.set_xlabel("Iteration")
        ax3.grid(True)

    else:
        fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))

        ax1.plot(iters, cells, marker='o')
        ax1.set_ylabel("Cell count")
        ax1.grid(True)

        ax2.plot(iters, areas, marker='s')
        ax2.set_ylabel("Area (um^2)")
        ax2.set_xlabel("Iteration")
        ax2.grid(True)

        fig.text(
            0.5, 0.01,
            "Logic-level data unavailable in current Yosys setup",
            ha="center", fontsize=10
        )

    plt.suptitle(f"LLM-Yosys Optimization Trajectory ({mode_name})")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    out_pdf = f"ppa_trajectory_{mode_name}.pdf"
    plt.savefig(out_pdf)
    plt.close()
    print(f"Saved {out_pdf}")

modes = ["area", "delay", "balanced"]

for mode in modes:
    log_file = f"optimization_log_{mode}.json"
    if os.path.exists(log_file):
        plot_one(log_file, mode)
    else:
        print(f"Missing {log_file}")

Writing plot_ppa_all.py


In [48]:
!python plot_ppa_all.py

Saved ppa_trajectory_area.pdf
Saved ppa_trajectory_delay.pdf
Saved ppa_trajectory_balanced.pdf
